# Uber Fares 🚙🚙
In this exercise, we'll use Random Forests in order to estimate the price of a Uber ride.

## Importing libraries and dataset
0. Import the usual libraries and read the dataset from this url:
"https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/Decision+trees/uber.csv"

In [1]:
# Step 0 — Importing libraries and dataset

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print("Loading dataset...")
url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/Decision+trees/uber.csv"
df = pd.read_csv(url)
print("...Done.\n")

print("Number of rows:", len(df))
df.head()


Loading dataset...
...Done.

Number of rows: 20000


,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,48462598,2015-05-07 10:24:44.0000004,13.0,2015-05-07 10:24:44 UTC,-73.971664,40.797035,-73.958939,40.777649,1
1,6637611,2014-07-09 09:14:04.0000002,5.5,2014-07-09 09:14:04 UTC,-73.991635,40.749855,-73.988250,40.741341,2
2,8357193,2013-11-11 18:51:00.000000240,8.5,2013-11-11 18:51:00 UTC,-73.982352,40.777042,-73.995912,40.759757,1
3,40466112,2014-05-22 01:54:00.00000069,19.0,2014-05-22 01:54:00 UTC,-73.991455,40.751700,-73.936357,40.812327,1
4,35405035,2011-06-21 23:37:33.0000002,7.7,2011-06-21 23:37:33 UTC,-73.974749,40.756255,-73.952276,40.778332,1


## Basic exploring and cleaning
1. Display basic statistics about the dataset. Do you notice some inconsistent values?

In [2]:
# Step — Display basic statistics

print("Number of rows :", len(df), "\n")

print("Preview of dataset:\n")
display(df.head())

print("\nBasic statistics:\n")
display(df.describe(include="all"))

print("\nPercentage of missing values:\n")
display(df.isna().mean() * 100)


Number of rows : 20000 

Preview of dataset:



,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,48462598,2015-05-07 10:24:44.0000004,13.0,2015-05-07 10:24:44 UTC,-73.971664,40.797035,-73.958939,40.777649,1
1,6637611,2014-07-09 09:14:04.0000002,5.5,2014-07-09 09:14:04 UTC,-73.991635,40.749855,-73.988250,40.741341,2
2,8357193,2013-11-11 18:51:00.000000240,8.5,2013-11-11 18:51:00 UTC,-73.982352,40.777042,-73.995912,40.759757,1
3,40466112,2014-05-22 01:54:00.00000069,19.0,2014-05-22 01:54:00 UTC,-73.991455,40.751700,-73.936357,40.812327,1
4,35405035,2011-06-21 23:37:33.0000002,7.7,2011-06-21 23:37:33 UTC,-73.974749,40.756255,-73.952276,40.778332,1



Basic statistics:



,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2.000000e+04,20000,20000.000000,20000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
unique,NaN,20000,NaN,19967,NaN,NaN,NaN,NaN,NaN
top,NaN,2010-06-10 23:28:00.000000172,NaN,2012-10-10 10:37:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,NaN,2,NaN,NaN,NaN,NaN,NaN
mean,2.767949e+07,NaN,11.358151,NaN,-72.490431,39.918498,-72.459891,39.923345,1.690150
std,1.601123e+07,NaN,9.891990,NaN,10.461597,6.051561,10.564266,6.901520,1.311384
min,3.949000e+03,NaN,-23.700000,NaN,-75.419276,-74.006190,-75.423067,-73.991765,0.000000
25%,1.383476e+07,NaN,6.000000,NaN,-73.992075,40.734733,-73.991423,40.734105,1.000000
50%,2.769724e+07,NaN,8.500000,NaN,-73.981904,40.752554,-73.980305,40.752997,1.000000
75%,4.148082e+07,NaN,12.500000,NaN,-73.967229,40.767075,-73.963509,40.768348,2.000000



Percentage of missing values:



,0
Unnamed: 0,0.0
key,0.0
fare_amount,0.0
pickup_datetime,0.0
pickup_longitude,0.0
pickup_latitude,0.0
dropoff_longitude,0.0
dropoff_latitude,0.0
passenger_count,0.0


2. Drop the useless columns and the rows containing outliers.

In [3]:
# Step — Drop useless columns and remove outliers

print("Dropping useless columns...")
df_clean = df.drop(columns=["Unnamed: 0", "key"])
# aucune valeur pour la modélisation
print("...Done.")



print("\nDropping rows with outliers in fare_amount...")

# tarifs positifs uniquement
df_clean = df_clean[df_clean["fare_amount"] > 0]


# 6 places max
df_clean = df_clean[df_clean["passenger_count"].between(1, 6)]

print("...Done.")
print("Number of rows after cleaning:", len(df_clean))

df_clean.describe()


Dropping useless columns...
...Done.

Dropping rows with outliers in fare_amount...
...Done.
Number of rows after cleaning: 19920


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,19920.000000,19920.000000,19920.000000,19920.000000,19920.000000,19920.000000
mean,11.365588,-72.488168,39.917188,-72.457523,39.922057,1.696687
std,9.898295,10.469574,6.056872,10.572571,6.909379,1.309701
min,0.010000,-75.419276,-74.006190,-75.423067,-73.991765,1.000000
25%,6.000000,-73.992063,40.734717,-73.991426,40.734107,1.000000
50%,8.500000,-73.981902,40.752543,-73.980318,40.752986,1.000000
75%,12.500000,-73.967229,40.767035,-73.963535,40.768326,2.000000
max,220.000000,40.803672,41.366138,40.831932,493.533332,6.000000


## Feature engineering
### Dealing with datetime objects
3. Convert the `pickup_datetime` column into datetime format. Use panda's [dt module](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.html) to create the following columns:
* Year
* Month
* Day
* Weekday: contains the **name** of the day of week

Then, you can drop the column `pickup_datetime`.

In [4]:


print("Converting pickup_datetime to datetime format...")

#  datetime sur df_clean
df_clean["pickup_datetime"] = pd.to_datetime(df_clean["pickup_datetime"], errors="coerce")

# il vaut mieux supprimer les dates invalides et non les imputer
# df_clean = df_clean.dropna(subset=["pickup_datetime"])

# year month, day, day_name
df_clean["year"] = df_clean["pickup_datetime"].dt.year
df_clean["month"] = df_clean["pickup_datetime"].dt.month
df_clean["day"] = df_clean["pickup_datetime"].dt.day
df_clean["weekday"] = df_clean["pickup_datetime"].dt.day_name()   # Monday, Tuesday...

print("...Done.")

# suppression de la colonne orginale
df_clean = df_clean.drop(columns=["pickup_datetime"])

#
df_clean.head()


Converting pickup_datetime to datetime format...
...Done.


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day,weekday
0,13.0,-73.971664,40.797035,-73.958939,40.777649,1,2015,5,7,Thursday
1,5.5,-73.991635,40.749855,-73.988250,40.741341,2,2014,7,9,Wednesday
2,8.5,-73.982352,40.777042,-73.995912,40.759757,1,2013,11,11,Monday
3,19.0,-73.991455,40.751700,-73.936357,40.812327,1,2014,5,22,Thursday
4,7.7,-73.974749,40.756255,-73.952276,40.778332,1,2011,6,21,Tuesday


### Haversine formula

It would be very interesting to compute the ride distance from the GPS coordinates. [Haversine formula](https://en.wikipedia.org/wiki/Haversine_formula) allows to do this 🤓:

$$
d = 2r \arcsin \big(\sqrt{\sin^2(\frac{\phi_2 - \phi_1}{2}) + \cos \phi_1 \cos \phi_2 \sin^2(\frac{\lambda_2 - \lambda_1}{2})} \big)
$$

where:
* $d$ is the ride distance in kilometers
* $r$ is the Earth's radius in kilometers
* $\phi_1$ is the pickup latitude in radians
* $\phi_2$ is the dropoff latitude in radians
* $\lambda_1$ is the pickup longitude in radians
* $\lambda_2$ is the dropoff longitude in radians

We've implemented for you a function that computes this formula for one ride with coordinates `lon_1`, `lon_2`, `lat_1` and `lat_2`:

In [5]:
def haversine(lon_1, lon_2, lat_1, lat_2):

    lon_1, lon_2, lat_1, lat_2 = map(np.radians, [lon_1, lon_2, lat_1, lat_2])  # Convert degrees to Radians
    # applique np.radians à tous les objets de la liste

    diff_lon = lon_2 - lon_1
    diff_lat = lat_2 - lat_1


    distance_km = 2*6371*np.arcsin(np.sqrt(np.sin(diff_lat/2.0)**2 + np.cos(lat_1) * np.cos(lat_2) * np.sin(diff_lon/2.0)**2)) # earth radius: 6371km

    return distance_km

4. Apply the `haversine` function to he whole dataset to create a new column `ride_distance`. [This stackoverflow post](https://stackoverflow.com/questions/13331698/how-to-apply-a-function-to-two-columns-of-pandas-dataframe?answertab=trending#tab-top) might help you!

In [6]:
print("New column creation")

df_clean["ride_distance"] = df_clean.apply(
    lambda row: haversine(
        row["pickup_longitude"],
        row["dropoff_longitude"],
        row["pickup_latitude"],
        row["dropoff_latitude"]
    ),
          # fonction anonyme qui prend chaque ligne du df
    #     def compute_distance(row):
    #       return haversine(
        #     row["pickup_longitude"],
        #     row["dropoff_longitude"],
        #     row["pickup_latitude"],
        #     row["dropoff_latitude"]
    # )
    axis=1 # applique la fonction sur chaque ligne
)

print("...Done.")

df_clean[["pickup_longitude", "pickup_latitude",
          "dropoff_longitude", "dropoff_latitude",
          "ride_distance"]].head()


New column creation
...Done.


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,ride_distance
0,-73.971664,40.797035,-73.958939,40.777649,2.407225
1,-73.991635,40.749855,-73.988250,40.741341,0.988729
2,-73.982352,40.777042,-73.995912,40.759757,2.235651
3,-73.991455,40.751700,-73.936357,40.812327,8.183379
4,-73.974749,40.756255,-73.952276,40.778332,3.099698


## Preprocessing
5. Separate the target from the features

In [7]:
print("Separating labels from features...")

y = df_clean["fare_amount"]
X = df_clean.drop(columns=["fare_amount"])

Separating labels from features...


6. Detect names of numeric/categorical features

In [8]:
# Step — Detect names of numeric and categorical features

numeric_features = X.select_dtypes(include=["int", "float"]).columns.tolist() # toujours int et float sinon pas de month etc...
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Found numeric features ", numeric_features)
print("Found categorical features ", categorical_features)


Found numeric features  ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year', 'month', 'day', 'ride_distance']
Found categorical features  ['weekday']


7. Make a train/test splitting with test_size = 0.2

In [9]:
from sklearn.model_selection import train_test_split

print("Dividing into train and test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
)

print("...Done.\n")

print("Shapes:")
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)


Dividing into train and test sets...
...Done.

Shapes:
X_train : (15936, 10)
X_test  : (3984, 10)
y_train : (15936,)
y_test  : (3984,)


8. Make all the necessary preprocessings.

Hint: in this exercise, we'll first create a baseline model with a multivariate **linear regression**. So don't forget to make all the transformations that are required for this kind of model 😉

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

print("Performing preprocessings on train set...")

# Preprocessor:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Fit on training data + transform
X_train_preprocessed = preprocessor.fit_transform(X_train)

print("...Done.\n")
print(X_train_preprocessed[:5])   # preview

print("\nPerforming preprocessings on test set...")

# Transform test data only (no fit!)
X_test_preprocessed = preprocessor.transform(X_test)

print("...Done.\n")
print(X_test_preprocessed[:5])


Performing preprocessings on train set...
...Done.

[[-0.14480146  0.13275414 -0.14574401  0.11625917  0.23594769 -0.40109631
   1.37463289  0.26843798 -0.0443416   1.          0.          0.
   0.          0.          0.          0.        ]
 [-0.14282184  0.13556986 -0.14227659  0.12389819  0.23594769 -0.93791253
   0.79673333 -1.3478406  -0.03280516  0.          0.          1.
   0.          0.          0.          0.        ]
 [-0.14315292  0.14114798 -0.14417842  0.11673196  0.23594769  0.67253614
  -1.51486492  1.30747421 -0.04767919  0.          0.          0.
   1.          0.          0.          0.        ]
 [-0.14395675  0.13792347 -0.14394303  0.1163065   1.76879011 -0.40109631
  -0.93696536  1.53837115 -0.04926102  0.          0.          0.
   0.          0.          1.          0.        ]
 [-0.13975619  0.14285037 -0.1445061   0.11506389  2.53521132 -1.47472875
  -1.51486492 -1.3478406  -0.03929802  0.          0.          0.
   1.          0.          0.          0.   

## Baseline: Linear Regression
9. Train a linear regression model and evaluate its performances. Is it satisfying?

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("Training linear regression model...")

linreg = LinearRegression()
linreg.fit(X_train_preprocessed, y_train)

print("...Done.\n")

# Predictions
y_train_pred = linreg.predict(X_train_preprocessed)
y_test_pred = linreg.predict(X_test_preprocessed)

# R2 scores
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print("R2 score on training set : ", r2_train)
print("R2 score on test set : ", r2_test)



Training linear regression model...
...Done.

R2 score on training set :  0.02332924287003535
R2 score on test set :  0.02111663761455962


## Random Forest
10. Train a Random Forest model with default hyperparameters. Are the performances better?

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

print("Training Random Forest with default hyperparameters...")

rf_default = RandomForestRegressor(random_state=42)
rf_default.fit(X_train_preprocessed, y_train)

print("...Done.\n")

# Predictions
y_train_pred_rf = rf_default.predict(X_train_preprocessed)
y_test_pred_rf = rf_default.predict(X_test_preprocessed)

# R2 scores
r2_train_rf = r2_score(y_train, y_train_pred_rf)
r2_test_rf = r2_score(y_test, y_test_pred_rf)

print("R2 score on training set :", r2_train_rf)
print("R2 score on test set :", r2_test_rf)



Training Random Forest with default hyperparameters...
...Done.

R2 score on training set : 0.9679821550736274
R2 score on test set : 0.7948462713447277


### Grid search
11. Use grid search to tune the model's hyperparameters. You can try the following values:

```
params = {
    'max_depth': [10, 12, 14],
    'min_samples_split': [4, 8],
    'n_estimators': [60, 80, 100]
}
```



In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

print("Grid search...")

params = {
    'max_depth': [10, 12, 14],
    'min_samples_split': [4, 8],
    'n_estimators': [60, 80, 100]
}

rf = RandomForestRegressor(random_state=42)

grid = GridSearchCV(
    estimator=rf,
    param_grid=params,
    scoring="r2",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_preprocessed, y_train)

print("...Done.\n")

print("Best hyperparameters :", grid.best_params_)
print("Best validation accuracy :", grid.best_score_)


Grid search...
Fitting 3 folds for each of 18 candidates, totalling 54 fits
...Done.

Best hyperparameters : {'max_depth': 10, 'min_samples_split': 4, 'n_estimators': 100}
Best validation accuracy : 0.7653956774811531


### Performances
12. Display the R2-score and the [mean absolute error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html?highlight=mean%20absolute%20error#sklearn.metrics.mean_absolute_error) on train set and test set. What do you think of this model? Would it be interesting to use it to estimate the fares on new data?

In [14]:
from sklearn.metrics import r2_score, mean_absolute_error

print("Evaluating optimized Random Forest model...")

# Best estimator from the grid search
best_rf = grid.best_estimator_

# Predictions
y_train_pred_opt = best_rf.predict(X_train_preprocessed)
y_test_pred_opt = best_rf.predict(X_test_preprocessed)

# R2 scores
r2_train_opt = r2_score(y_train, y_train_pred_opt)
r2_test_opt = r2_score(y_test, y_test_pred_opt)

# MAE scores
mae_train_opt = mean_absolute_error(y_train, y_train_pred_opt)
mae_test_opt = mean_absolute_error(y_test, y_test_pred_opt)

print("R2 score on training set :", r2_train_opt)
print("R2 score on test set :", r2_test_opt)

print("\nMean Absolute Error on training set :", mae_train_opt)
print("Mean Fare on training set :", y_train.mean())

print("\nMean Absolute Error on test set :", mae_test_opt)
print("Mean Fare on test set :", y_test.mean())
print("Standard-deviation on test set :", y_test.std())


Evaluating optimized Random Forest model...
R2 score on training set : 0.8976499368269506
R2 score on test set : 0.7905059209907385

Mean Absolute Error on training set : 1.8068925217610683
Mean Fare on training set : 11.385798820281124

Mean Absolute Error on test set : 2.1121091691932743
Mean Fare on test set : 11.284743975903615
Standard-deviation on test set : 9.78190274564962


## Feature importance
13. Make a bar plot with the importances of each feature. Are you surprised?

14. Would the model be able to make good predictions if we hadn't included the ride distance by hand? Train a new Random Forest model (with grid search) by dropping the `ride_distance` column from the features, and conclude.